<a href="https://colab.research.google.com/github/TU-USUARIO/labo1-colabs/blob/main/01_Primeros_pasos_medir_y_reportar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Colab 01 — Primeros pasos: medir, leer el instrumento y graficar

**Laboratorio 1 · Clase 1**

**Objetivos.** Al terminar este notebook vas a poder:

1. Ejecutar código en Colab y guardar tu propia copia.
2. Leer un **calibre** y un **micrómetro**, y entender de dónde sale la resolución de cada uno (O1.2).
3. Medir y corregir el **error de cero** de un instrumento (O1.3).
4. Elegir el instrumento adecuado para cada dimensión y justificar la elección (O1.4).
5. Distinguir **resolución** de **dispersión**, y aplicar la regla operativa que usa la cátedra (O1.5).
6. Distinguir una lista de Python de un arreglo de NumPy, y hacer un gráfico presentable (O1.6).

**Requisitos previos:** ninguno.
**Tiempo estimado:** 55 minutos.

> **Antes de empezar:** hacé `Archivo → Guardar una copia en Drive`. Vas a trabajar sobre *tu* copia; el original queda intacto para el resto del curso.

> **Todavía no vamos a hablar de propagación ni de redondeo del par (valor, incerteza).** Eso es del Colab 02, donde aparece el primer resultado calculado. Acá el foco está en **leer bien un instrumento**, que es la operación sobre la que se apoya todo el resto del cuatrimestre.

---
## 1. El entorno

Un notebook es una secuencia de **celdas**. Las hay de dos tipos: de texto (como ésta) y de código
(como la de abajo). Para ejecutar una celda de código hacé clic en ella y apretá `Shift + Enter`.

Dos cosas que conviene saber desde el principio:

- Las celdas comparten memoria y se ejecutan **en el orden en que vos las corrés**, no en el orden
  en que están escritas. Si algo no funciona, `Entorno de ejecución → Reiniciar y ejecutar todo`
  resuelve el 90 % de los problemas raros. Ese es, además, el estado en el que se entrega un notebook.
- Todo lo que escribas acá se pierde si no guardás tu copia en Drive.

In [ ]:
# Primera celda: ejecutala con Shift+Enter
print("Todo listo.")

# Python funciona como una calculadora
2 + 2

In [ ]:
# Variables: un nombre que apunta a un valor
longitud   = 12.70      # mm
apreciacion = 0.05      # mm  (resolución del calibre)
print(longitud, apreciacion)
print(type(longitud))

---
## 2. Los tres instrumentos

### 2.1 La regla

Escala grabada cada 1 mm. La resolución es 1 mm, y con algo de práctica se puede *apreciar* la mitad
de la división. Nada más: si escribís 12,73 mm leído con regla, estás inventando la tercera cifra.

### 2.2 El calibre: el principio del nonio

El nonio es una idea de 1631 y es más ingeniosa de lo que parece. Sobre la mordaza móvil hay una
segunda escala con **$n$ divisiones que ocupan exactamente $n-1$ divisiones de la escala principal**.

Consecuencia: cada división del nonio mide $\frac{n-1}{n}$ de una división principal, así que la
diferencia entre una división principal y una del nonio es

$$ \text{resolución} = \frac{D}{n} $$

donde $D$ es el paso de la escala principal. Con $D = 1$ mm y $n = 20$ divisiones, la resolución es
0,05 mm. Con $n = 50$, es 0,02 mm.

**Cómo se lee.** El cero del nonio indica los milímetros enteros sobre la escala principal. Después
se busca **cuál de las rayas del nonio coincide** con una raya de la escala principal: el número de
esa raya, multiplicado por la resolución, es la fracción de milímetro.

### 2.3 El micrómetro: el tornillo y el trinquete

Acá la resolución no viene de una escala sino del **paso del tornillo**: típicamente 0,5 mm por
vuelta. El tambor tiene 50 divisiones, así que cada división vale $0{,}5/50 = 0{,}01$ mm.

Y una pieza que suele ignorarse y es la más importante desde el punto de vista metodológico: el
**trinquete**. Existe porque la lectura de un micrómetro depende de con cuánta fuerza aprietes, y esa
fuerza es una variable libre del experimentador. El trinquete la fija. Si medís sin usarlo, cada
integrante del grupo va a obtener un número distinto y la culpa no va a ser del instrumento.

> **Regla que se lleva escrita al cuaderno:** el instrumento más fino no sirve de nada si el objeto no
> está definido a esa escala. Con un micrómetro de 0,01 mm sobre una pieza rugosa vas a medir la
> rugosidad, no el diámetro.

In [ ]:
def resolucion_nonio(paso_principal_mm, n_divisiones):
    '''Resolución de un calibre con nonio de n divisiones.'''
    return paso_principal_mm / n_divisiones

for n in (10, 20, 50):
    print(f"nonio de {n:2d} divisiones  ->  resolución = "
          f"{resolucion_nonio(1.0, n):.3f} mm")

print()
paso_tornillo, div_tambor = 0.5, 50     # mm por vuelta, divisiones
print(f"micrómetro: paso {paso_tornillo} mm / {div_tambor} divisiones "
      f"-> resolución = {paso_tornillo/div_tambor:.3f} mm")

---
## 3. Lo que hace el instrumento con el valor verdadero

Un instrumento con resolución $\Delta$ no te devuelve el valor verdadero: te devuelve el múltiplo de
$\Delta$ más cercano. Eso se llama **cuantización**, y es la primera fuente de incerteza del curso.

La celda de abajo toma un objeto de longitud verdadera fija y simula qué reporta cada instrumento.

In [ ]:
import numpy as np

def cuantizar(valor, resolucion):
    '''Lo que reporta un instrumento de resolución dada.'''
    return np.round(valor / resolucion) * resolucion

L_verdadero = 12.7134     # mm  (en la vida real no lo conocés; acá sí, para ver qué pasa)

instrumentos = {"regla      (1 mm)":    1.00,
                "calibre    (0,05 mm)": 0.05,
                "micrómetro (0,01 mm)": 0.01}

print(f"longitud verdadera: {L_verdadero} mm\n")
for nombre, delta in instrumentos.items():
    leido = cuantizar(L_verdadero, delta)
    print(f"{nombre:<22s}: lee {leido:8.3f} mm   (error de cuantización = "
          f"{leido - L_verdadero:+.4f} mm)")

Ahora la pregunta que importa: si el objeto pudiera tener cualquier longitud, ¿cómo se distribuye ese
error de cuantización?

In [ ]:
import matplotlib.pyplot as plt

rng = np.random.default_rng(20260812)
delta = 0.05                                   # calibre
verdaderos = rng.uniform(10, 15, size=20000)   # objetos de longitud cualquiera
errores = cuantizar(verdaderos, delta) - verdaderos

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.hist(errores, bins=40, density=True, edgecolor='k', alpha=0.75)
ax.set_xlabel('Error de cuantización [mm]')
ax.set_ylabel('Densidad')
ax.set_title(f'El error de resolución es UNIFORME en $\\pm\\Delta/2$   ($\\Delta$ = {delta} mm)')
ax.grid(alpha=0.3)
fig.tight_layout(); plt.show()

print(f"ancho del intervalo : {delta} mm")
print(f"desviación estándar : {errores.std(ddof=1):.5f} mm")
print(f"Δ/√12               : {delta/np.sqrt(12):.5f} mm   <- guardá este número")

El histograma es un **rectángulo**, no una campana. Esto es importante y se retoma en el Colab 03: no
todo error es gaussiano, y el error de resolución es el contraejemplo más común. Su desviación
estándar es $\Delta/\sqrt{12} \approx 0{,}29\,\Delta$, y ése es el número que después vamos a usar
como $\sigma$ cuando la única fuente de incerteza sea la resolución del instrumento.

Por ahora, guardalo. En el Colab 02 lo vamos a usar sin haberlo justificado del todo, y ahí se avisa.

---
## 4. El error de cero

Antes de medir cualquier cosa: **cerrá el instrumento y leé**. Si no marca cero, todas tus mediciones
van a estar corridas por esa misma cantidad. Eso es un **error sistemático**, y tiene dos propiedades
que conviene tener claras desde hoy:

- **No se reduce midiendo más veces.** Está en todas las mediciones por igual.
- **Se corrige restándolo**, siempre que lo hayas medido. Por eso se mide y se anota antes que nada.

In [ ]:
# Ejemplo: calibre que con las mordazas cerradas marca +0,10 mm
error_cero = 0.10                                        # mm
lecturas_crudas = np.array([12.80, 12.85, 12.75, 12.80, 12.85])   # mm

lecturas = lecturas_crudas - error_cero

print("crudas    :", lecturas_crudas)
print("corregidas:", np.round(lecturas, 3))
print()
print(f"promedio crudo      : {lecturas_crudas.mean():.3f} mm")
print(f"promedio corregido  : {lecturas.mean():.3f} mm")
print(f"la diferencia es exactamente el error de cero: {error_cero} mm")

> **Ejercicio 1.1.** El promedio de las lecturas crudas y el de las corregidas difieren en 0,10 mm,
> pero la **dispersión** de las dos series es idéntica. Verificalo con `np.std(lecturas_crudas, ddof=1)`
> y `np.std(lecturas, ddof=1)`. ¿Por qué mirar la dispersión no te habría permitido detectar el
> error de cero?

---
## 5. Resolución no es lo mismo que incerteza

Éste es el punto de la Clase 1 y sale de tus propios datos, no de este notebook.

Cuando todos los integrantes del grupo miden lo mismo con el **mismo** instrumento, la dispersión
entre ellos no tiene por qué coincidir con la resolución:

- Con la **regla**, la dispersión suele ser del orden de la resolución. El instrumento es el que manda.
- Con el **micrómetro**, la dispersión suele ser **varias veces** la resolución. Ahí lo que manda ya
  no es el instrumento: es el objeto (que no es perfectamente cilíndrico) y el observador.

> **Regla operativa de la cátedra:** la incerteza de una medición directa es **la mayor** entre la
> resolución del instrumento y la dispersión observada al repetir.

Cargá abajo las mediciones reales de tu grupo y miralo.

In [ ]:
# --- REEMPLAZAR POR LOS DATOS DE TU GRUPO -----------------------------
medidas = {
    "regla      (Δ = 1 mm)":    np.array([13.0, 12.5, 13.0, 12.5, 13.0]),
    "calibre    (Δ = 0,05 mm)": np.array([12.70, 12.75, 12.70, 12.65, 12.70]),
    "micrómetro (Δ = 0,01 mm)": np.array([12.703, 12.741, 12.688, 12.729, 12.696]),
}
resoluciones = [1.0, 0.05, 0.01]
# ----------------------------------------------------------------------

print(f"{'instrumento':26s} {'promedio':>10s} {'dispersión':>12s} "
      f"{'resolución':>12s} {'manda':>12s}")
print("-" * 76)
for (nombre, x), delta in zip(medidas.items(), resoluciones):
    disp = np.std(x, ddof=1)
    manda = "dispersión" if disp > delta else "resolución"
    print(f"{nombre:26s} {x.mean():10.4f} {disp:12.4f} {delta:12.4f} {manda:>12s}")

---
## 6. Listas y arreglos

Python trae listas. NumPy trae arreglos (`ndarray`). Se parecen, pero **no se comportan igual** en
las operaciones aritméticas, y esa diferencia es la razón por la que en física usamos NumPy.

In [ ]:
lista   = [1.0, 2.0, 3.0, 4.0]
arreglo = np.array(lista)

print("lista   * 2 :", lista * 2)      # ¡repite la lista!
print("arreglo * 2:", arreglo * 2)     # multiplica cada elemento

In [ ]:
# Con arreglos, las operaciones se aplican elemento a elemento sin escribir un solo bucle
diametros = np.array([12.71, 12.68, 12.74, 12.70, 12.69])   # mm
radios    = diametros / 2
areas     = np.pi * radios**2                                # mm^2

print("radios:", radios)
print("áreas :", np.round(areas, 3))
print("cantidad de datos:", len(diametros), " | media:", diametros.mean())

Esto se llama **vectorización**. Además de ser más corto y más rápido, se parece mucho más a la
fórmula del papel, así que es más fácil de revisar — que es lo que más importa en un informe.

Formas útiles de crear arreglos:

In [ ]:
print(np.zeros(5))                 # cinco ceros
print(np.arange(0, 1.1, 0.25))     # de 0 a 1 en pasos de 0,25
print(np.linspace(0, 1, 5))        # 5 puntos equiespaciados entre 0 y 1 (incluye ambos extremos)

`linspace` es la que vas a usar casi siempre para graficar una curva teórica: le pedís muchos puntos
en el rango de tus datos y evaluás el modelo ahí.

---
## 7. El primer gráfico

Un gráfico de laboratorio tiene, como mínimo:

- ejes rotulados **con la magnitud y su unidad**;
- barras de error si las hay (y casi siempre las hay);
- puntos como puntos —los datos son puntos, no una línea— y curvas como líneas;
- tamaño de letra que se lea proyectado.

El error más común en los informes de Laboratorio 1 es unir los datos experimentales con líneas
rectas. No lo hagas: esa línea afirma que mediste lo que hay entre dos puntos, y no lo mediste.

De las **siete reglas de graficación** de la materia, hoy aplicamos cuatro. Las otras tres (línea
del modelo, gráfico de residuos, escala logarítmica justificada) necesitan un ajuste, y el primer
ajuste es de la Clase 4.

In [ ]:
# Las mediciones del grupo, una por integrante, con el mismo instrumento
integrante = np.array([1, 2, 3, 4, 5])
valor      = medidas["micrómetro (Δ = 0,01 mm)"]
err        = np.full_like(valor, 0.01)        # resolución del micrómetro

fig, ax = plt.subplots(figsize=(6.4, 4))
ax.errorbar(integrante, valor, yerr=err, fmt='o', capsize=3, label='mediciones')
ax.axhline(valor.mean(), color='crimson', ls='--', lw=1.4,
           label=f'promedio = {valor.mean():.3f} mm')
ax.set_xlabel('Integrante del grupo')
ax.set_ylabel('Diámetro $D$ [mm]')
ax.set_title('Mismo objeto, mismo instrumento, cinco observadores')
ax.set_xticks(integrante)
ax.grid(alpha=0.3); ax.legend()
fig.tight_layout(); plt.show()

Mirá las barras de error: son la resolución del instrumento, y son **más chicas que la dispersión
entre observadores** — los puntos no se solapan dentro de sus barras. Ése es exactamente el
diagnóstico de la tabla anterior, ahora en forma de figura, y es la señal de que la resolución no es
la incerteza que corresponde.

Rehacé el mismo gráfico con los datos de la regla y vas a ver lo contrario: todos los puntos caen
dentro de la barra del vecino. Ahí la resolución sí alcanza.

Para guardar la figura y pegarla en el informe:

In [ ]:
fig.savefig('diametro_por_observador.pdf', bbox_inches='tight')   # PDF: vectorial, no pixela
fig.savefig('diametro_por_observador.png', dpi=200, bbox_inches='tight')
# Aparecen en el panel de archivos (ícono de carpeta, a la izquierda). Botón derecho → Descargar.
print("figuras guardadas")

---
## 8. Leer tus propios datos

En Colab hay tres maneras de traer un archivo:

1. **Subirlo a mano:** panel de archivos (carpeta a la izquierda) → botón de subir. Rápido, pero se
   borra cuando se reinicia el entorno.
2. **Montar tu Drive:** `from google.colab import drive; drive.mount('/content/drive')`. Persistente.
3. **Con el diálogo de subida**, que es lo que hace la celda de abajo.

Guardá tus mediciones en un `.txt` o `.csv` con una columna por magnitud. Nada de celdas combinadas
ni títulos en el medio de los datos.

In [ ]:
# Descomentá para subir un archivo desde tu computadora
# from google.colab import files
# subidos = files.upload()

# Y para leerlo (una columna por magnitud, separadas por espacios o comas):
# datos = np.loadtxt('mis_datos.txt')
# x, y = datos[:, 0], datos[:, 1]

---
## 9. Ejercicios

**1.2.** Medí el mismo objeto con la regla, el calibre y el micrómetro. Para cada instrumento anotá
la resolución y la dispersión entre los integrantes del grupo, y decidí cuál de los dos números es la
incerteza. ¿Con cuál instrumento conviene medir este objeto, y por qué el más fino no es
automáticamente la respuesta?

**1.3.** Medí el error de cero de tu calibre y de tu micrómetro cinco veces cada uno. ¿El error de
cero es reproducible? Si no lo es, ¿qué significa eso, y cómo lo incorporarías a la incerteza?

**1.4.** Graficá tus mediciones individuales (número de medición en el eje x, valor en el eje y) con
una línea horizontal en el promedio. ¿Se ve alguna tendencia? Si el valor creciera sistemáticamente
con el número de medición, ¿qué estarías detectando? (Pista: el micrómetro se calienta con la mano.)

**1.5.** Volvé a la Sección 3 y cambiá `delta` a 0,01 mm. ¿Cambia la *forma* del histograma del error
de cuantización? ¿Y su desviación estándar? Verificá que la relación $\sigma = \Delta/\sqrt{12}$ se
mantiene.

**1.6.** *(conceptual)* Un compañero te dice que midió con micrómetro y por eso su incerteza es
0,01 mm. Sus cinco mediciones son 12,70; 12,74; 12,69; 12,76; 12,71. ¿Está bien lo que dice?
Escribí en dos oraciones qué le contestarías.